# Cross-Category Purchase Patterns ETL

## Purpose
Identify which departments are frequently purchased together in the same order for cross-category merchandising and bundling strategies.

## Input
* **Source:** `big_data.silver.order_products`
* **Source:** `big_data.silver.products_enriched` 

## Output
* **Target:** `big_data.gold.vw_cross_category_patterns`
* **Refresh:** Real-time (always reflects current Silver data)

## SQL Logic
1. JOIN order_products with products_enriched to get department
2. Self-join on order_id where departments differ
3. Get distinct department pairs
4. COUNT orders per department pair
5. ORDER BY order count DESC

In [0]:
%sql
-- Cross-Category Purchase Patterns View
-- Purpose: Identify department pairs frequently purchased together

CREATE OR REPLACE VIEW big_data.gold.vw_cross_category_patterns AS
WITH enriched AS (
  SELECT 
    op.order_id,
    op.product_id,
    p.department
  FROM big_data.silver.order_products op
  LEFT JOIN big_data.silver.products_enriched p ON op.product_id = p.product_id
),
dept_pairs AS (
  SELECT DISTINCT
    a.department AS dept1,
    b.department AS dept2
  FROM enriched a
  JOIN enriched b ON a.order_id = b.order_id
  WHERE a.department != b.department
)
SELECT 
  dept1,
  dept2,
  COUNT(*) AS order_count
FROM dept_pairs
GROUP BY dept1, dept2
ORDER BY order_count DESC;

In [0]:
%sql
-- Verify view exists and preview top department pairs

SELECT * FROM big_data.gold.vw_cross_category_patterns
LIMIT 10;